## Basic prompting

In [3]:
from pathlib import Path
import sys

from dotenv import load_dotenv

def find_project_root():
    for path in (Path.cwd(), *Path.cwd().parents):
        if (path / "minimax_model.py").exists():
            return path
    raise FileNotFoundError("Could not find minimax_model.py")

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

load_dotenv(PROJECT_ROOT / ".env", override=True)

True

In [4]:
from langchain.agents import create_agent
from minimax_model import get_model
from langchain.messages import HumanMessage

agent = create_agent(model=get_model())

question = HumanMessage(content="What's the capital of the moon?")

response = agent.invoke(
    {"messages": [question]}
)

print(response['messages'][1].content)

<think>
The user asks: "What's the capital of the moon?" This is a straightforward question: presumably no official capital of the Moon. The moon is a natural satellite, not a country, has no government. So answer: The Moon has no capital; it's not a sovereign state. Or maybe talk about "The Moon" in sci-fi? Usually answer: There's no capital; no recognized government. Could mention that the "Moon" has no official capital; the United Nations doesn't have a governing body; the Moon is governed by the Outer Space Treaty. There's some speculation about a "Luna City" in the future. Or "Moon's capital" in a fictional context? But the question is likely a test for us to know that the Moon has no capital. So answer accordingly. Perhaps mention that the Moon is considered a celestial body under international law, governed by the Outer Space Treaty, which prohibits national appropriation. So no capital. Also mention that some proposals for lunar bases such as "Artemis Base Camp" might have a na

In [5]:
from minimax_model import get_model
system_prompt = "You are a science fiction writer, create a capital city at the users request."

scifi_agent = create_agent(
    model=get_model(),
    system_prompt=system_prompt
)

response = scifi_agent.invoke(
    {"messages": [question]}
)

print(response['messages'][1].content)

<think>The user is asking about the capital of the moon. This is a bit of a trick question. There's currently no official capital of the moon because it's not an independent nation or political entity. However, in science fiction, authors often create capitals for lunar settlements.

Let me think about this from a science fiction perspective, since the user mentioned I'm a science fiction writer. I should be creative and give an interesting answer.

In real life, there's no capital of the moon because:
1. The moon isn't a sovereign nation
2. There's no formal government or capital city on the moon
3. The moon is governed by the Outer Space Treaty of 1967, which says no nation can claim sovereignty over it

In science fiction, I've seen various creative takes on lunar capitals. For example:
- In some sci-fi universes, "Lunapolis" is often used as a name
- Some stories have capitals inside lunar bases or domed cities
- The Moon is sometimes depicted as having a separate government from E

## Few-shot examples

In [6]:
from minimax_model import get_model
system_prompt = """

You are a science fiction writer, create a space capital city at the users request.

User: What is the capital of mars?
Scifi Writer: Marsialis

User: What is the capital of Venus?
Scifi Writer: Venusovia

"""

scifi_agent = create_agent(
    model=get_model(),
    system_prompt=system_prompt
)

response = scifi_agent.invoke(
    {"messages": [question]}
)

print(response['messages'][1].content)

<think>
The user is asking about the capital of the moon, continuing the sci-fi worldbuilding game. I should create a fitting name for a lunar capital city that sounds appropriately futuristic and lunar-themed.

Let me come up with something creative:

Some options:
- Lunaria
- Selenopolis
- Moonspire
- Lunar Prime
- Selen City
- Tranquility Base (could reference Apollo 11)
- Artemis City
- Lunaxis

I think "Selenopolis" has a nice ring to it - "selene" is Greek for moon, "polis" means city. Or "Lunaria" which sounds elegant. Let me go with something that feels both futuristic and lunar.

I'll go with **Selenopolis** - it sounds like a proper capital city name for the moon.
</think>

**Selenopolis**


## Structured prompts

In [7]:
from minimax_model import get_model
system_prompt = """

You are a science fiction writer, create a space capital city at the users request.

Please keep to the below structure.

Name: The name of the capital city

Location: Where it is based

Vibe: 2-3 words to describe its vibe

Economy: Main industries

"""

scifi_agent = create_agent(
    model=get_model(),
    system_prompt=system_prompt
)

response = scifi_agent.invoke(
    {"messages": [question]}
)

print(response['messages'][1].content)

<think>
The user is asking me to create a fictional space capital city for the moon. This is a creative writing request, so I'll make up an interesting sci-fi capital city following the structure they provided.

Let me create something compelling that feels like it belongs in a well-thought-out science fiction universe.
</think>

**Name:** Selene Prime

**Location:** Crater Tycho, southern lunar highlands

**Vibe:** Quiet, luminous, bureaucratic

**Economy:** Helium-3 extraction, off-world treaty administration, orbital transit logistics

---

A city grown into the ancient walls of a impact crater, Selene Prime hums with the low-level activity of a thousand bureaucratic errands. Solar panels blanket the outer slopes while the city itself sprawls beneath the regolith, a lattice of pressurized corridors and glass biodomes where Earth hangs perpetually in the black sky. Helium-3 shipments leave weekly for the fusion stations of Mars and Ceres, but most residents spend their days filing ma

## Structured output

In [8]:
from langchain.agents import create_agent
from minimax_model import get_model
from langchain.messages import HumanMessage
from pydantic import BaseModel

class CapitalInfo(BaseModel):
    name: str
    location: str
    vibe: str
    economy: str

agent = create_agent(
    model=get_model(),
    system_prompt="You are a science fiction writer, create a capital city at the users request.",
    response_format=CapitalInfo
)

question = HumanMessage(content="What is the capital of The Moon?")

response = agent.invoke(
    {"messages": [question]}
)

response["structured_response"]

CapitalInfo(name='Lunaris Prime', location='Central lunar basin, strategically positioned for equitable access from all settlement regions', vibe='A gleaming techno-utopia blending ancient lunar mystique with cutting-edge innovation; bustling with scientists, miners, diplomats, and dreamers from across the solar system', economy='Lunar mining operations, interplanetary trade, advanced research facilities, and tourism')

In [12]:
response["structured_response"].location

'Central lunar basin, strategically positioned for equitable access from all settlement regions'

In [11]:
capital_info = response["structured_response"]

capital_name = capital_info.name
capital_location = capital_info.location

print(f"{capital_name} is a city located at {capital_location}")

Lunaris Prime is a city located at Central lunar basin, strategically positioned for equitable access from all settlement regions
